# GwenLand glcuda - T4 Ceiling Wave 11

Clean Wave 4 factorial sprint for Tesla T4: **A = exact adjacent glue fusion**, **B = exact GQA7 K/V reuse**. The four production arms are ordered by a balanced Williams square. Wave 10 remains historical evidence and is not stacked into this candidate.

The bootstrap reuses Cargo when present and installs the official minimal Rust toolchain when Kaggle omits it. Hard gates: pinned model/base/patch hashes, `ptxas sm_75`, zero spills, >=24 projected resident warps, byte-identical fused-kernel/GQA parity, exact `50/50` greedy oracle, four production sessions per arm, and >=5% improvement in every paired block. Microbenchmarks and NCU are diagnostic only.

## 1 - Bootstrap, patch stack, and structural contracts

In [ ]:
import ast
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import time
import traceback
import urllib.error
import urllib.request
import zipfile

NOTEBOOK_BUILD = "wave11-exact-fusion-gqa7-v2-cargo-bootstrap"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE11_PATCH_SHA256 = "0e644ae9c37ecdf73fd354a8c59100ed31bb3edce5c76e13b5f08ef4dd3292ca"
WAVE11_PATCH_GZIP_B64 = """..."""

HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491400032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
TARGET_PREFILL_TPS = 15000.0
PRODUCTION_REPEATS = 4
COLD_ITERS = 3
WARMUP_ITERS = 3
MEASURE_ITERS = 10
RUN_NCU = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave11-{RUN_ID}"
META_REPO = ROOT / "meta"
RESULTS = ROOT / "results"
ARM_DIRS = {"wave4": ROOT / "wave4", "wave11": ROOT / "wave11"}
ARM_TARGETS = {name: ROOT / f"target-{name}" for name in ARM_DIRS}
FINAL_ZIP = WORK / "glcuda_t4_ceiling_wave11_fetch_results.zip"
ROOT.mkdir(parents=True, exist_ok=False)
RESULTS.mkdir(parents=True)

def run(cmd, cwd=None, env=None, timeout=1800, check=True):
    merged = os.environ.copy()
    # A reused Kaggle kernel may carry flags from an earlier experiment.  Each
    # child process starts from a controlled glcuda dispatch environment so the
    # factorial differs only by the values explicitly supplied below.
    for key in [
        "GLCUDA_FORCE_Q8", "GLCUDA_GRID2D", "GLCUDA_R256", "GLCUDA_NO_MMA",
        "GLCUDA_FUSE_Q8_GLUE", "GLCUDA_GQA_GROUP", "GLCUDA_MULTI_STREAM_PREFILL",
        "GLCUDA_PROFILE_PREFILL", "GLCUDA_PROFILE_DECODE", "GLCUDA_TELEMETRY",
        "GLCUDA_CACHE", "GLCUDA_JIT_VERBOSE",
    ]:
        merged.pop(key, None)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=merged,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout,
    )
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}")
    return p

def save_log(name, proc):
    (RESULTS / name).write_text(
        f"returncode={proc.returncode}\n\nSTDOUT\n{proc.stdout}\n\nSTDERR\n{proc.stderr}",
        encoding="utf-8",
    )


def ensure_cargo():
    cargo_home = WORK / ".wave11-cargo"
    rustup_home = WORK / ".wave11-rustup"
    candidates = [
        shutil.which("cargo"),
        cargo_home / "bin/cargo",
        Path.home() / ".cargo/bin/cargo",
        Path("/usr/local/cargo/bin/cargo"),
    ]
    cargo = next((Path(x) for x in candidates if x and Path(x).is_file()), None)
    if cargo is None:
        installer = ROOT / "rustup-init"
        url = "https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init"
        last_error = None
        for attempt in range(1, 6):
            try:
                request = urllib.request.Request(
                    url,
                    headers={"User-Agent": "GwenLand-glcuda-Wave11/1.1", "Accept-Encoding": "identity"},
                )
                with urllib.request.urlopen(request, timeout=120) as response:
                    payload = response.read()
                if len(payload) < (1 << 20):
                    raise RuntimeError(f"rustup-init download is unexpectedly small: {len(payload)} bytes")
                installer.write_bytes(payload)
                installer.chmod(0o755)
                break
            except Exception as exc:
                last_error = exc
                if attempt == 5:
                    raise RuntimeError(f"cannot download rustup-init: {last_error}") from exc
                time.sleep(min(30, 2 ** attempt))
        install_env = {
            "CARGO_HOME": str(cargo_home),
            "RUSTUP_HOME": str(rustup_home),
            "PATH": f"{cargo_home / 'bin'}:{os.environ.get('PATH', '')}",
        }
        install = run(
            [installer, "-y", "--profile", "minimal", "--default-toolchain", "stable", "--no-modify-path"],
            env=install_env,
            timeout=1800,
        )
        save_log("rustup-init.log", install)
        cargo = cargo_home / "bin/cargo"
    cargo_bin = cargo.parent
    os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    if cargo_home in cargo.parents:
        os.environ["CARGO_HOME"] = str(cargo_home)
        os.environ["RUSTUP_HOME"] = str(rustup_home)
    cargo = Path(shutil.which("cargo") or cargo)
    rustc = shutil.which("rustc")
    if not cargo.is_file() or not rustc:
        raise RuntimeError(f"Rust toolchain bootstrap incomplete: cargo={cargo}, rustc={rustc}")
    cargo_version = run([cargo, "--version"])
    rustc_version = run([rustc, "--version", "--verbose"])
    save_log("rust-toolchain.log", cargo_version)
    with (RESULTS / "rust-toolchain.log").open("a", encoding="utf-8") as f:
        f.write(f"\n\nRUSTC\n{rustc_version.stdout}\n{rustc_version.stderr}")
    print(f"Rust toolchain: {cargo_version.stdout.strip()} / {rustc_version.stdout.splitlines()[0]}")
    return str(cargo)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def partial_archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    return FINAL_ZIP

def fail_phase(phase):
    text = traceback.format_exc()
    (RESULTS / "FAILED.json").write_text(json.dumps({"phase": phase, "traceback": text}, indent=2), encoding="utf-8")
    partial_archive()
    raise RuntimeError(f"Wave 11 {phase} failed; partial archive: {FINAL_ZIP}\n{text}")

def decode_patch(encoded, expected, label):
    data = gzip.decompress(base64.b64decode(encoded))
    got = hashlib.sha256(data).hexdigest()
    if got != expected:
        raise RuntimeError(f"{label} patch hash mismatch: {got} != {expected}")
    path = RESULTS / f"{label}.patch"
    path.write_bytes(data)
    return path

assert callable(time.monotonic)
PATCHES = {
    "wave3": decode_patch(WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256, "wave3"),
    "wave4": decode_patch(WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256, "wave4"),
    "wave11": decode_patch(WAVE11_PATCH_GZIP_B64, WAVE11_PATCH_SHA256, "wave11"),
}

try:
    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"], timeout=60)
    rows = [x.strip() for x in gpu.stdout.splitlines() if x.strip()]
    if not rows:
        raise RuntimeError("nvidia-smi returned no GPU")
    fields = [x.strip() for x in rows[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"Wave 11 requires Tesla T4 sm_75, got {rows[0]}")
    GPU_INFO = {"raw": rows[0], "index": fields[0], "name": fields[1], "compute_cap": fields[2], "memory_mib": fields[3], "driver": fields[4]}
    (RESULTS / "gpu.json").write_text(json.dumps(GPU_INFO, indent=2), encoding="utf-8")
    CARGO = ensure_cargo()

    clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
    save_log("git-clone.log", clone)
    obj = run(["git", "cat-file", "-t", BASE_REV], cwd=META_REPO, check=False)
    if obj.returncode:
        fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
        save_log("git-fetch-base.log", fetch)
        obj = run(["git", "cat-file", "-t", BASE_REV], cwd=META_REPO)
    if obj.stdout.strip() != "commit":
        raise RuntimeError(f"BASE_REV is not a commit: {obj.stdout!r}")

    for name, path in ARM_DIRS.items():
        run(["git", "worktree", "add", "--detach", path, BASE_REV], cwd=META_REPO)
        stack = [PATCHES["wave3"], PATCHES["wave4"]] + ([PATCHES["wave11"]] if name == "wave11" else [])
        for patch in stack:
            run(["git", "apply", "--whitespace=error", patch], cwd=path)
        run(["git", "diff", "--check"], cwd=path)

    candidate_ptx = (ARM_DIRS["wave11"] / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
    for entry in ["gl_rms_quantize_q8_rows", "gl_silu_mul_quantize_q8", "gl_attn_decode_rows_gqa7_f32"]:
        if f".visible .entry {entry}(" not in candidate_ptx:
            raise RuntimeError(f"missing Wave 11 PTX entry {entry}")
    if "cp.async" in candidate_ptx:
        raise RuntimeError("sm_75 candidate contains forbidden cp.async")
    baseline_ptx = (ARM_DIRS["wave4"] / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
    if any(x in baseline_ptx for x in ["gl_rms_quantize_q8_rows", "gl_attn_decode_rows_gqa7_f32"]):
        raise RuntimeError("Wave 4 control contains Wave 11 entry points")
    STACK_OK = True
    print(f"Wave 11 stack ready at {ROOT}")
    print(json.dumps(GPU_INFO, indent=2))
except Exception:
    fail_phase("bootstrap")

## 2 - Resumable pinned model fetch

In [ ]:
if not globals().get("STACK_OK"):
    raise RuntimeError("Bootstrap gate did not pass")

def fetch_pinned_model():
    model = WORK / HF_FILENAME
    part = WORK / f"{HF_FILENAME}.part"
    if model.is_file() and model.stat().st_size == HF_EXPECTED_BYTES and sha256_file(model) == HF_EXPECTED_SHA256:
        return model
    if model.exists():
        model.unlink()
    if part.exists():
        part_size = part.stat().st_size
        if part_size == HF_EXPECTED_BYTES:
            if sha256_file(part) == HF_EXPECTED_SHA256:
                part.replace(model)
                return model
            part.unlink()
        elif part_size > HF_EXPECTED_BYTES:
            part.unlink()
    url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"
    for attempt in range(1, 6):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "GwenLand-glcuda-Wave11/1.0", "Accept-Encoding": "identity"}
        if start:
            headers["Range"] = f"bytes={start}-"
        request = urllib.request.Request(url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
            status = getattr(response, "status", response.getcode())
            if start and status != 206:
                response.close()
                part.unlink(missing_ok=True)
                start = 0
                response = urllib.request.urlopen(
                    urllib.request.Request(url, headers={"User-Agent": "GwenLand-glcuda-Wave11/1.0", "Accept-Encoding": "identity"}),
                    timeout=120,
                )
                status = getattr(response, "status", response.getcode())
            if status not in (200, 206):
                raise RuntimeError(f"HTTP {status}")
            mode = "ab" if start and status == 206 else "wb"
            downloaded = start
            last_print = time.monotonic()
            with response, part.open(mode) as output:
                while True:
                    chunk = response.read(8 << 20)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if time.monotonic() - last_print >= 20:
                        print(f"model fetch {downloaded / (1 << 20):.1f}/{HF_EXPECTED_BYTES / (1 << 20):.1f} MiB")
                        last_print = time.monotonic()
            if part.stat().st_size != HF_EXPECTED_BYTES:
                raise RuntimeError(f"truncated model: {part.stat().st_size}/{HF_EXPECTED_BYTES}")
            digest = sha256_file(part)
            if digest != HF_EXPECTED_SHA256:
                part.unlink(missing_ok=True)
                raise RuntimeError(f"model SHA mismatch: {digest}")
            part.replace(model)
            return model
        except Exception as exc:
            print(f"fetch attempt {attempt}/5 failed: {exc}")
            if attempt == 5:
                raise
            time.sleep(min(30, 2 ** attempt))

try:
    MODEL_PATH = fetch_pinned_model()
    MODEL_META = {"repo": HF_REPO, "revision": HF_REVISION, "filename": HF_FILENAME, "bytes": MODEL_PATH.stat().st_size, "sha256": sha256_file(MODEL_PATH)}
    (RESULTS / "model.json").write_text(json.dumps(MODEL_META, indent=2), encoding="utf-8")
    MODEL_OK = True
    print(json.dumps(MODEL_META, indent=2))
except Exception:
    fail_phase("model-fetch")

## 3 - Build, ptxas, zero-spill, and occupancy gates

In [ ]:
if not globals().get("MODEL_OK"):
    raise RuntimeError("Model gate did not pass")

def parse_ptxas(text):
    resources = {}
    current = None
    for line in text.splitlines():
        m = re.search(r"Function properties for\s+'?([^'\s]+)'?", line)
        if m:
            current = m.group(1)
            resources.setdefault(current, {"spill_stores": 0, "spill_loads": 0})
        if current:
            m = re.search(r"(\d+) bytes spill stores, (\d+) bytes spill loads", line)
            if m:
                resources[current]["spill_stores"] = int(m.group(1))
                resources[current]["spill_loads"] = int(m.group(2))
            m = re.search(r"Used\s+(\d+) registers(?:,\s*(\d+) bytes smem)?", line)
            if m:
                resources[current]["registers"] = int(m.group(1))
                resources[current]["smem_bytes"] = int(m.group(2) or 0)
    return resources

def occupancy(resource, threads, dynamic=0):
    regs = max(1, int(resource.get("registers", 0)))
    smem = int(resource.get("smem_bytes", 0)) + dynamic
    limits = [16, 1024 // threads, 65536 // (regs * threads)]
    if smem:
        limits.append(65536 // smem)
    ctas = max(0, min(limits))
    return {"ctas": ctas, "warps": ctas * (threads // 32), "threads": threads, "dynamic_smem": dynamic}

try:
    PTXAS = shutil.which("ptxas")
    if not PTXAS:
        raise RuntimeError("ptxas not found in Kaggle CUDA image")
    BUILD_INFO = {}
    for name, path in ARM_DIRS.items():
        target = ARM_TARGETS[name]
        env = {"CARGO_TARGET_DIR": str(target)}
        check = run([CARGO, "check", "-p", "glcuda", "--locked"], cwd=path, env=env, timeout=3600)
        save_log(f"cargo-check-{name}.log", check)
        tests = run([CARGO, "test", "-p", "glcuda", "--lib", "--locked"], cwd=path, env=env, timeout=3600)
        save_log(f"cargo-lib-{name}.log", tests)
        build_glbench = run([CARGO, "build", "--release", "-p", "glbench", "--locked"], cwd=path, env=env, timeout=7200)
        save_log(f"cargo-build-{name}.log", build_glbench)
        BUILD_INFO[name] = {"glbench": str(target / "release/glbench")}
    diag_build = run([CARGO, "build", "--release", "-p", "glcuda", "--example", "wave11", "--locked"], cwd=ARM_DIRS["wave11"], env={"CARGO_TARGET_DIR": str(ARM_TARGETS["wave11"])}, timeout=7200)
    save_log("cargo-build-wave11-diag.log", diag_build)
    BUILD_INFO["wave11"]["diag"] = str(ARM_TARGETS["wave11"] / "release/examples/wave11")

    PTXAS_RESOURCES = {}
    for name, path in ARM_DIRS.items():
        for module in ["glcuda.ptx", "glcuda_sm75.ptx"]:
            src = path / "glcuda/src/kernels" / module
            out = RESULTS / f"{name}-{module}.cubin"
            p = run([PTXAS, "-arch=sm_75", "-v", "--warn-on-spills", src, "-o", out], timeout=1800, check=False)
            save_log(f"ptxas-{name}-{module}.log", p)
            if p.returncode:
                raise RuntimeError(f"ptxas failed for {name}/{module}")
            parsed = parse_ptxas(p.stdout + "\n" + p.stderr)
            PTXAS_RESOURCES[f"{name}/{module}"] = parsed
            if any(v.get("spill_stores", 0) or v.get("spill_loads", 0) for v in parsed.values()):
                raise RuntimeError(f"spill detected in {name}/{module}: {parsed}")

    new = PTXAS_RESOURCES["wave11/glcuda.ptx"]
    required = {
        "gl_rms_quantize_q8_rows": (256, 0),
        "gl_silu_mul_quantize_q8": (256, 0),
        "gl_attn_decode_rows_gqa7_f32": (128, 8944),
    }
    OCCUPANCY = {}
    for entry, (threads, dynamic) in required.items():
        if entry not in new or "registers" not in new[entry]:
            raise RuntimeError(f"ptxas resource record missing {entry}: {new}")
        OCCUPANCY[entry] = occupancy(new[entry], threads, dynamic)
        if OCCUPANCY[entry]["warps"] < 24:
            raise RuntimeError(f"Wave 11 occupancy gate failed for {entry}: {new[entry]} / {OCCUPANCY[entry]}")
    (RESULTS / "ptxas-resources.json").write_text(json.dumps(PTXAS_RESOURCES, indent=2), encoding="utf-8")
    (RESULTS / "occupancy.json").write_text(json.dumps(OCCUPANCY, indent=2), encoding="utf-8")
    BUILD_OK = True
    print(json.dumps(OCCUPANCY, indent=2))
except Exception:
    fail_phase("build-ptxas-resource")

## 4 - Hardware exactness and diagnostic microbenchmarks

In [ ]:
if not globals().get("BUILD_OK"):
    raise RuntimeError("Build/resource gate did not pass")
try:
    PARITY = {}
    for name, path in ARM_DIRS.items():
        env = {"CARGO_TARGET_DIR": str(ARM_TARGETS[name]), "CUDA_VISIBLE_DEVICES": "0"}
        if name == "wave11":
            env.update({"GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_GQA_GROUP": "1"})
        p = run([CARGO, "test", "--release", "-p", "glcuda", "--test", "parity", "--locked", "--", "--test-threads=1", "--nocapture"], cwd=path, env=env, timeout=7200, check=False)
        save_log(f"hardware-parity-{name}.log", p)
        hay = p.stdout + "\n" + p.stderr
        if p.returncode or "SKIP:" in hay:
            raise RuntimeError(f"hardware parity failed or skipped for {name}")
        if name == "wave11" and not all(x in hay for x in ["wave11_fused_rms_q8_is_bit_exact", "wave11_fused_silu_q8_is_bit_exact", "wave11_gqa7_is_bit_exact"]):
            raise RuntimeError("Wave 11 exact kernel tests were not executed")
        PARITY[name] = "PASS"

    diag = run([BUILD_INFO["wave11"]["diag"]], cwd=ARM_DIRS["wave11"], env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=7200, check=False)
    save_log("wave11-diagnostic.log", diag)
    if diag.returncode:
        raise RuntimeError("Wave 11 diagnostic example failed")
    match = re.search(r"\[wave11-diag\]\s*(\{.*\})", diag.stdout)
    if not match:
        raise RuntimeError("Wave 11 diagnostic JSON missing")
    DIAGNOSTIC = json.loads(match.group(1))
    if DIAGNOSTIC.get("warmup") != 10 or DIAGNOSTIC.get("iters") != 100:
        raise RuntimeError(f"diagnostic iteration contract failed: {DIAGNOSTIC}")
    (RESULTS / "diagnostic.json").write_text(json.dumps(DIAGNOSTIC, indent=2), encoding="utf-8")
    CORRECTNESS_OK = True
    print(json.dumps({"parity": PARITY, "diagnostic": DIAGNOSTIC}, indent=2))
except Exception:
    fail_phase("hardware-correctness")

## 5 - Balanced Williams production factorial

In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise RuntimeError("Hardware correctness gate did not pass")

prompt_unit = "Measure this deterministic systems prompt carefully. Explain how token-parallel integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
FIXED_PROMPT = prompt_unit * 8
ARMS = [
    # Keep one binary across the 2x2 factorial.  With both Wave 11 flags off,
    # dispatch follows the retained Wave 4 hot path; using the same executable
    # also balances module load/JIT and gives every arm the runtime JSON contract.
    ("wave4_attn_dsmem", "wave11", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1"}, {"exact_fusion": False, "gqa_group": False, "grid2d": True, "r256": False}),
    ("wave11_exact_fusion", "wave11", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_FUSE_Q8_GLUE": "1"}, {"exact_fusion": True, "gqa_group": False, "grid2d": True, "r256": False}),
    ("wave11_gqa_group", "wave11", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_GQA_GROUP": "1"}, {"exact_fusion": False, "gqa_group": True, "grid2d": True, "r256": False}),
    ("wave11_combined", "wave11", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_GQA_GROUP": "1"}, {"exact_fusion": True, "gqa_group": True, "grid2d": True, "r256": False}),
]
ARM_MAP = {label: (build_name, env, contract) for label, build_name, env, contract in ARMS}
WILLIAMS_LABELS = [
    ["wave4_attn_dsmem", "wave11_exact_fusion", "wave11_combined", "wave11_gqa_group"],
    ["wave11_exact_fusion", "wave11_gqa_group", "wave4_attn_dsmem", "wave11_combined"],
    ["wave11_gqa_group", "wave11_combined", "wave11_exact_fusion", "wave4_attn_dsmem"],
    ["wave11_combined", "wave4_attn_dsmem", "wave11_gqa_group", "wave11_exact_fusion"],
]
assert all(sorted(row) == sorted(ARM_MAP) for row in WILLIAMS_LABELS)
assert all(len({row[p] for row in WILLIAMS_LABELS}) == 4 for p in range(4))
assert len({(row[i], row[i + 1]) for row in WILLIAMS_LABELS for i in range(3)}) == 12

def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] * (hi - index) + values[hi] * (index - lo)

def contract_from_log(hay):
    matches = re.findall(r"\[glcuda-contract\]\s*(\{[^\n]+\})", hay)
    if not matches:
        raise RuntimeError("machine-readable glcuda contract missing")
    return json.loads(matches[-1])

def session_stats(path, expected_iters=MEASURE_ITERS):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    engine = data.get("engine") or {}
    workload = data.get("workload") or {}
    if engine.get("name") != "glcuda" or engine.get("backend") != "cuda" or not engine.get("available"):
        raise RuntimeError(f"wrong engine contract: {engine}")
    expected_workload = {
        "engine": "glcuda", "kind": "prefill", "prompt": FIXED_PROMPT,
        "seed": 42, "temperature": 0.0, "max_new_tokens": 1,
        "cold_iters": COLD_ITERS, "warmup_iters": WARMUP_ITERS,
        "measure_iters": expected_iters, "verify_against": "glproc",
    }
    for key, value in expected_workload.items():
        if workload.get(key) != value:
            raise RuntimeError(f"workload {key} mismatch: {workload.get(key)!r} != {value!r}")
    if Path(workload.get("model_path", "")).name != HF_FILENAME:
        raise RuntimeError(f"wrong model path: {workload.get('model_path')}")
    validation = data.get("validation") or {}
    if validation.get("passed") is not True:
        raise RuntimeError(f"session validation failed: {validation}")
    parity = [f for f in validation.get("findings", []) if f.get("check") == "parity"]
    if not parity or parity[-1].get("severity") != "info":
        raise RuntimeError(f"oracle parity evidence missing: {validation}")
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", ""))
    if not match or match.group(1) != match.group(2) or int(match.group(2)) != 50:
        raise RuntimeError(f"exact 50/50 oracle failed: {parity[-1] if parity else None}")
    iterations = (data.get("measurements") or {}).get("iterations") or []
    if len(iterations) != expected_iters:
        raise RuntimeError(f"wrong measured iteration count: {len(iterations)}")
    prompt_counts = [int(x.get("prompt_tokens", 0)) for x in iterations]
    prefill_ms = [float(x.get("prefill_ms", 0)) for x in iterations]
    decode_ms = [float(x.get("decode_ms", 0)) for x in iterations]
    if len(set(prompt_counts)) != 1 or prompt_counts[0] != 244 or any(x <= 0 for x in prefill_ms + decode_ms):
        raise RuntimeError(f"malformed iteration payload: {iterations}")
    tps = [prompt_counts[0] * 1000.0 / x for x in prefill_ms]
    decode_tps = [1000.0 / x for x in decode_ms]
    return {
        "prompt_tokens": prompt_counts[0], "prefill_p50": percentile(tps, .5),
        "prefill_p90": percentile(tps, .9), "prefill_p95": percentile(tps, .95),
        "prefill_p99": percentile(tps, .99), "prefill_mean": statistics.mean(tps),
        "latency_p95_ms": percentile(prefill_ms, .95), "latency_p99_ms": percentile(prefill_ms, .99),
        "decode_p50": percentile(decode_tps, .5), "oracle": "50/50",
    }

def gpu_snapshot(repeat, position, arm, phase):
    query = "timestamp,index,name,pstate,temperature.gpu,power.draw,clocks.current.sm,clocks.current.memory,utilization.gpu,memory.used"
    p = run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits", "-i", "0"], timeout=60, check=False)
    return {"repeat": repeat, "position": position, "arm": arm, "phase": phase, "returncode": p.returncode, "csv": p.stdout.strip()}

def glbench_cmd(out, cold, warmup, iters):
    return ["run", "--engine", "glcuda", "--model", MODEL_PATH, "--prompt", FIXED_PROMPT, "--tokens", "1", "--cold-iters", str(cold), "--warmup", str(warmup), "--iters", str(iters), "--temperature", "0", "--seed", "42", "--kind", "prefill", "--verify-against", "glproc", "--out", out]

try:
    # One discarded stabilization run per arm after all builds/correctness checks.
    for label, build_name, extra_env, expected_contract in ARMS:
        out = RESULTS / f"stabilize-{label}.json"
        p = run([BUILD_INFO[build_name]["glbench"], *glbench_cmd(out, 0, 1, 1)], cwd=ARM_DIRS[build_name], env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
        save_log(f"stabilize-{label}.log", p)
        if p.returncode:
            raise RuntimeError(f"stabilization failed: {label}")
        got = contract_from_log(p.stdout + "\n" + p.stderr)
        if {k: got.get(k) for k in expected_contract} != expected_contract:
            raise RuntimeError(f"dispatch truth-table mismatch for {label}: {got}")
        if "[glcuda] GLCUDA_FORCE_Q8:" not in p.stdout + "\n" + p.stderr:
            raise RuntimeError(f"forced-Q8 loader contract missing for {label}")

    PROD_RECORDS = []
    HARDWARE_SNAPSHOTS = []
    for repeat, order in enumerate(WILLIAMS_LABELS):
        for position, label in enumerate(order):
            build_name, extra_env, expected_contract = ARM_MAP[label]
            archive = RESULTS / f"glbench-{repeat}-{position}-{label}.json"
            HARDWARE_SNAPSHOTS.append(gpu_snapshot(repeat, position, label, "before"))
            p = run([BUILD_INFO[build_name]["glbench"], *glbench_cmd(archive, COLD_ITERS, WARMUP_ITERS, MEASURE_ITERS)], cwd=ARM_DIRS[build_name], env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
            HARDWARE_SNAPSHOTS.append(gpu_snapshot(repeat, position, label, "after"))
            save_log(f"glbench-{repeat}-{position}-{label}.log", p)
            if p.returncode:
                raise RuntimeError(f"glbench failed: {label} repeat {repeat}")
            got = contract_from_log(p.stdout + "\n" + p.stderr)
            if {k: got.get(k) for k in expected_contract} != expected_contract:
                raise RuntimeError(f"dispatch truth-table mismatch for {label}: {got}")
            if "[glcuda] GLCUDA_FORCE_Q8:" not in p.stdout + "\n" + p.stderr:
                raise RuntimeError(f"forced-Q8 loader contract missing for {label}")
            stats = session_stats(archive)
            PROD_RECORDS.append({"repeat": repeat, "position": position, "arm": label, "archive": str(archive), **stats})
            print(f"{label:22s} r{repeat} p{position}: {stats['prefill_p50']:.1f} tok/s | P95 {stats['latency_p95_ms']:.2f} ms | oracle {stats['oracle']}")

    PROD_SUMMARY = []
    for label in ARM_MAP:
        rows = [x for x in PROD_RECORDS if x["arm"] == label]
        PROD_SUMMARY.append({
            "arm": label,
            "session_p50_median": statistics.median(x["prefill_p50"] for x in rows),
            "session_mean_median": statistics.median(x["prefill_mean"] for x in rows),
            "session_p90_median": statistics.median(x["prefill_p90"] for x in rows),
            "session_p95_median": statistics.median(x["prefill_p95"] for x in rows),
            "session_p99_median": statistics.median(x["prefill_p99"] for x in rows),
            "latency_p95_median_ms": statistics.median(x["latency_p95_ms"] for x in rows),
            "latency_p99_median_ms": statistics.median(x["latency_p99_ms"] for x in rows),
            "decode_p50_median": statistics.median(x["decode_p50"] for x in rows),
            "sessions": len(rows), "positions": [x["position"] for x in rows],
        })
    summary = {x["arm"]: x for x in PROD_SUMMARY}

    def paired_decision(candidate):
        paired = []
        for repeat in range(PRODUCTION_REPEATS):
            base = next(x for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == "wave4_attn_dsmem")
            cand = next(x for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == candidate)
            paired.append({
                "repeat": repeat,
                "prefill_p50_delta": cand["prefill_p50"] / base["prefill_p50"] - 1,
                "prefill_mean_delta": cand["prefill_mean"] / base["prefill_mean"] - 1,
                "p95_latency_delta": cand["latency_p95_ms"] / base["latency_p95_ms"] - 1,
                "p99_latency_delta": cand["latency_p99_ms"] / base["latency_p99_ms"] - 1,
                "decode_p50_delta": cand["decode_p50"] / base["decode_p50"] - 1,
            })
        median_delta = summary[candidate]["session_p50_median"] / summary["wave4_attn_dsmem"]["session_p50_median"] - 1
        result = {
            "candidate": candidate, "median_delta": median_delta, "paired": paired,
            "prefill_p50_pass": all(x["prefill_p50_delta"] >= .05 for x in paired),
            "prefill_mean_pass": all(x["prefill_mean_delta"] >= .05 for x in paired),
            "tail_pass": all(x["p95_latency_delta"] <= .05 and x["p99_latency_delta"] <= .05 for x in paired),
            "decode_pass": all(x["decode_p50_delta"] >= -.05 for x in paired),
            "oracle_pass": all(x["oracle"] == "50/50" for x in PROD_RECORDS if x["arm"] == candidate),
        }
        result["retain"] = median_delta >= .05 and all(result[k] for k in ["prefill_p50_pass", "prefill_mean_pass", "tail_pass", "decode_pass", "oracle_pass"])
        return result

    DECISIONS = {label: paired_decision(label) for label in list(ARM_MAP)[1:]}
    y00 = summary["wave4_attn_dsmem"]["session_p50_median"]
    y10 = summary["wave11_exact_fusion"]["session_p50_median"]
    y01 = summary["wave11_gqa_group"]["session_p50_median"]
    y11 = summary["wave11_combined"]["session_p50_median"]
    FACTORIAL = {
        "fusion_log_main_effect": math.exp((math.log(y10) + math.log(y11) - math.log(y00) - math.log(y01)) / 2) - 1,
        "gqa_log_main_effect": math.exp((math.log(y01) + math.log(y11) - math.log(y00) - math.log(y10)) / 2) - 1,
        "multiplicative_interaction": math.exp(math.log(y11) - math.log(y10) - math.log(y01) + math.log(y00)) - 1,
    }

    TELEMETRY = {}
    for label, (build_name, extra_env, _) in ARM_MAP.items():
        out = RESULTS / f"telemetry-{label}.json"
        p = run([BUILD_INFO[build_name]["glbench"], *glbench_cmd(out, 0, 3, 1)], cwd=ARM_DIRS[build_name], env={"CUDA_VISIBLE_DEVICES": "0", "GLCUDA_TELEMETRY": "1", **extra_env}, timeout=14400, check=False)
        save_log(f"telemetry-{label}.log", p)
        if p.returncode:
            raise RuntimeError(f"telemetry failed: {label}")
        data = json.loads(out.read_text(encoding="utf-8"))
        stages = (((data.get("telemetry") or {}).get("prefill") or {}).get("stages") or [])
        if not stages:
            raise RuntimeError(f"telemetry emitted no stages: {label}")
        TELEMETRY[label] = stages

    BEST_ARM = max(PROD_SUMMARY, key=lambda x: x["session_p50_median"])["arm"]
    best_tps = summary[BEST_ARM]["session_p50_median"]
    stages = TELEMETRY[BEST_ARM]
    stage_total = sum(float(x.get("total_ms") or 0) for x in stages)
    attention_ms = sum(float(x.get("total_ms") or 0) for x in stages if x.get("name") == "attention")
    gemm_ms = sum(float(x.get("total_ms") or 0) for x in stages if x.get("name") in {"qkv", "attn_out", "ffn_gate_up", "ffn_down"})
    TARGET_ANALYSIS = {
        "best_arm": BEST_ARM, "measured_tps": best_tps, "target_tps": TARGET_PREFILL_TPS,
        "required_speedup": TARGET_PREFILL_TPS / best_tps,
        "measured_prefill_ms": 244000.0 / best_tps, "target_prefill_ms": 244000.0 / TARGET_PREFILL_TPS,
        "attention_share": attention_ms / stage_total, "gemm_share": gemm_ms / stage_total,
        "infinite_attention_ceiling_tps": best_tps / (1 - attention_ms / stage_total),
        "infinite_gemm_ceiling_tps": best_tps / (1 - gemm_ms / stage_total),
    }
    (RESULTS / "hardware-snapshots.json").write_text(json.dumps(HARDWARE_SNAPSHOTS, indent=2), encoding="utf-8")
    (RESULTS / "production-records.json").write_text(json.dumps(PROD_RECORDS, indent=2), encoding="utf-8")
    (RESULTS / "production-summary.json").write_text(json.dumps(PROD_SUMMARY, indent=2), encoding="utf-8")
    (RESULTS / "factorial.json").write_text(json.dumps(FACTORIAL, indent=2), encoding="utf-8")
    (RESULTS / "decisions.json").write_text(json.dumps(DECISIONS, indent=2), encoding="utf-8")
    (RESULTS / "target-analysis.json").write_text(json.dumps(TARGET_ANALYSIS, indent=2), encoding="utf-8")
    PROD_OK = True
    print(json.dumps(PROD_SUMMARY, indent=2))
    print(json.dumps(FACTORIAL, indent=2))
    print(json.dumps(DECISIONS, indent=2))
except Exception:
    fail_phase("production-factorial")

## 6 - Optional Nsight Compute evidence

In [ ]:
if not globals().get("PROD_OK"):
    raise RuntimeError("Production gate did not complete")
try:
    NCU = {"available": False, "attempted": False, "permission_denied": False}
    ncu_bin = shutil.which("ncu")
    if RUN_NCU and ncu_bin:
        NCU.update({"available": True, "attempted": True})
        p = run([ncu_bin, "--set", "basic", "--target-processes", "all", BUILD_INFO["wave11"]["diag"]], cwd=ARM_DIRS["wave11"], env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=7200, check=False)
        save_log("ncu-wave11.log", p)
        hay = p.stdout + "\n" + p.stderr
        NCU["returncode"] = p.returncode
        NCU["permission_denied"] = "ERR_NVGPUCTRPERM" in hay
        if p.returncode and not NCU["permission_denied"]:
            NCU["error"] = hay[-4000:]
    (RESULTS / "ncu.json").write_text(json.dumps(NCU, indent=2), encoding="utf-8")
    NCU_OK = True
    print(json.dumps(NCU, indent=2))
except Exception:
    fail_phase("optional-ncu")

## 7 - Report and always-downloadable evidence archive

In [ ]:
if not globals().get("NCU_OK"):
    raise RuntimeError("NCU phase did not complete")
try:
    summary = {x["arm"]: x for x in PROD_SUMMARY}
    lines = [
        "# glcuda T4 Ceiling - Wave 11", "",
        f"- notebook: {NOTEBOOK_BUILD}", f"- GPU: {GPU_INFO['raw']}",
        f"- baseline revision: {BASE_REV}", f"- Wave 3 patch: {WAVE3_PATCH_SHA256}",
        f"- retained Wave 4 patch: {WAVE4_PATCH_SHA256}", f"- Wave 11 patch: {WAVE11_PATCH_SHA256}",
        f"- model: {HF_REPO}@{HF_REVISION}/{HF_FILENAME}", f"- model SHA-256: {HF_EXPECTED_SHA256}",
        "- ptxas: PASS", "- hardware exact-kernel correctness: PASS", "- production glbench: COMPLETE",
        f"- warmup/measure: {WARMUP_ITERS}/{MEASURE_ITERS}", f"- target: {TARGET_PREFILL_TPS:.0f} prefill tok/s", "",
        "## Diagnostic medians", "",
        "| chain | legacy us | candidate us | delta |", "|---|---:|---:|---:|",
        f"| RMSNorm + Q8 | {DIAGNOSTIC['rms_q8_legacy_us']:.1f} | {DIAGNOSTIC['rms_q8_fused_us']:.1f} | {(DIAGNOSTIC['rms_q8_fused_us']/DIAGNOSTIC['rms_q8_legacy_us']-1)*100:+.2f}% |",
        f"| SwiGLU + Q8 | {DIAGNOSTIC['silu_q8_legacy_us']:.1f} | {DIAGNOSTIC['silu_q8_fused_us']:.1f} | {(DIAGNOSTIC['silu_q8_fused_us']/DIAGNOSTIC['silu_q8_legacy_us']-1)*100:+.2f}% |",
        f"| attention | {DIAGNOSTIC['attn_legacy_us']:.1f} | {DIAGNOSTIC['attn_gqa7_us']:.1f} | {(DIAGNOSTIC['attn_gqa7_us']/DIAGNOSTIC['attn_legacy_us']-1)*100:+.2f}% |", "",
        "## Production prefill", "",
        "| arm | P50 tok/s | mean | P90 | P95 | P99 | P95 latency ms | decode P50 | sessions |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for row in PROD_SUMMARY:
        lines.append(f"| {row['arm']} | {row['session_p50_median']:.1f} | {row['session_mean_median']:.1f} | {row['session_p90_median']:.1f} | {row['session_p95_median']:.1f} | {row['session_p99_median']:.1f} | {row['latency_p95_median_ms']:.2f} | {row['decode_p50_median']:.1f} | {row['sessions']} |")
    lines += ["", "## Factorial effects (log scale)", ""]
    lines += [f"- exact-fusion main effect: {FACTORIAL['fusion_log_main_effect']*100:+.2f}%", f"- GQA main effect: {FACTORIAL['gqa_log_main_effect']*100:+.2f}%", f"- multiplicative interaction: {FACTORIAL['multiplicative_interaction']*100:+.2f}%", ""]
    lines += ["## Retention decisions", ""]
    for label, decision in DECISIONS.items():
        lines.append(f"- {label}: {decision['median_delta']*100:+.2f}% - {'RETAIN' if decision['retain'] else 'REJECT/HOLD'}")
    lines += ["", "## 15k feasibility budget", "", f"- best arm: {TARGET_ANALYSIS['best_arm']} at {TARGET_ANALYSIS['measured_tps']:.1f} tok/s", f"- required speedup: {TARGET_ANALYSIS['required_speedup']:.2f}x", f"- measured/target prompt time: {TARGET_ANALYSIS['measured_prefill_ms']:.2f} / {TARGET_ANALYSIS['target_prefill_ms']:.2f} ms", f"- attention/GEMM share: {TARGET_ANALYSIS['attention_share']*100:.1f}% / {TARGET_ANALYSIS['gemm_share']*100:.1f}%", f"- infinite-attention-only ceiling: {TARGET_ANALYSIS['infinite_attention_ceiling_tps']:.1f} tok/s", f"- infinite-GEMM-only ceiling: {TARGET_ANALYSIS['infinite_gemm_ceiling_tps']:.1f} tok/s", "", "## Interpretation rule", "", "Retain only when the candidate median and every paired Williams block improve P50 and mean by at least 5%, P95/P99 latency and decode stay within 5%, exact 50/50 oracle and byte-identical kernel parity remain green, and ptxas reports zero spills in the >=24-warp resource tier. Diagnostics are not retention evidence."]
    report = "\n".join(lines) + "\n"
    (RESULTS / "REPORT.md").write_text(report, encoding="utf-8")
    manifest = {
        "notebook": NOTEBOOK_BUILD, "status": "COMPLETE", "base_revision": BASE_REV,
        "patches": {"wave3": WAVE3_PATCH_SHA256, "wave4": WAVE4_PATCH_SHA256, "wave11": WAVE11_PATCH_SHA256},
        "model": MODEL_META, "gpu": GPU_INFO, "best_arm": TARGET_ANALYSIS["best_arm"],
        "decisions": DECISIONS, "ncu": NCU,
    }
    (RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    indexed = []
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file() and path.name != "artifact-index.sha256":
            indexed.append(f"{sha256_file(path)}  {path.relative_to(RESULTS).as_posix()}")
    (RESULTS / "artifact-index.sha256").write_text("\n".join(indexed) + "\n", encoding="utf-8")
    partial_archive()
    print(report)
    print(f"Results directory: {RESULTS}")
    print(f"Download archive: {FINAL_ZIP}")
    print(f"Archive size: {FINAL_ZIP.stat().st_size / (1 << 20):.2f} MB")
except Exception:
    fail_phase("package")